In [29]:
# Broadcasting
import numpy as np
cpu = np.array([42,55,78,91,63])  #shape(5,)
print(cpu.shape)
print("cpu : ", cpu)
print("cpu-50 : ",cpu - 50)
print("cpu > 75 : ", cpu > 75)

(5,)
cpu :  [42 55 78 91 63]
cpu-50 :  [-8  5 28 41 13]
cpu > 75 :  [False False  True  True False]


In [35]:
# 2D array and 1D array - per row operation
servers = np.array([
    [42,55,78,91,63],
    [30,35,40,38,45],
    [88,92,95,97,99],
])

thresholds = np.array([50,60,70,80,70]) #threshold per time step -shape(5,)
print("servers \n ", servers) # shape(3,5)
print("servers shape", servers.shape)

print("thresholds \n", thresholds) #shape(5,1)
print("thresholds shape", thresholds.shape)
diff = servers - thresholds # substracts threshold from each column across all rows. 
# no loop needed. matching 5-element threshold
print("\n  difference of servers and thresholds \n", diff)
print("shape of difference", diff.shape)

servers 
  [[42 55 78 91 63]
 [30 35 40 38 45]
 [88 92 95 97 99]]
servers shape (3, 5)
thresholds 
 [50 60 70 80 70]
thresholds shape (5,)

  difference of servers and thresholds 
 [[ -8  -5   8  11  -7]
 [-20 -25 -30 -42 -25]
 [ 38  32  25  17  29]]
shape of difference (3, 5)


In [5]:
# 2D array and a column vector - per server operation
servers = np.array([
    [42,55,78,91,63],
    [30,35,40,38,45],
    [88,92,95,97,99],
])
baselines = np.array([ [50], # serverA
                      [35], # serverB
                      [90]  # serverC
                    ])
# column vector broadcasts across all 5 columns
diff = servers - baselines
print(diff) 

[[-8  5 28 41 13]
 [-5  0  5  3 10]
 [-2  2  5  7  9]]


In [44]:
# Normalisation — a key AIOps use case
# Normalisation scales all readings to 0–1 so different metrics can be compared fairly. 
# CPU % and latency ms are on completely different scales — normalising puts them on the same footing.

servers = np.array([
    [42, 55, 78, 91, 63],
    [30, 35, 40, 38, 45],
    [88, 92, 95, 97, 99],
])

# min and max per server - keepdims keeps shape (3,1) not (3,)
row_min = servers.min(axis=1, keepdims=True)
row_max = servers.max(axis=1, keepdims = True)

print("row_min\n", row_min)
print(" row_max\n", row_max)
#normalise - broadcasting handles the (3,1) vs (3,5) mismatch
normalised = (servers - row_min) / (row_max - row_min)
print("normalised readings \n", normalised.round(2))

row_min
 [[42]
 [30]
 [88]]
 row_max
 [[91]
 [45]
 [99]]
normalised readings 
 [[0.   0.27 0.73 1.   0.43]
 [0.   0.33 0.67 0.53 1.  ]
 [0.   0.36 0.64 0.82 1.  ]]


In [45]:
# Z-score per server — broadcasting in anomaly detection
import numpy as np
servers = np.array([
    [42, 55, 78, 91, 63],
    [30, 35, 40, 38, 45],
    [88, 92, 95, 97, 99],
])
mean = servers.mean(axis=1, keepdims=True)
std = servers.std(axis=1, keepdims=True)
print(f"mean per server:\n {mean}")
print(f"std per server:\n {std}")

# z-score - how many std deviations from each servers mean
z = (servers - mean) / std
print("z sore \n", z.round(2))
print("Anamoly detection on z-score:\n", z > 1.5) # flag any thing above 1.5 sta deviations

# Each server is compared against its own mean and std — not a global threshold.
# Server B running at 45% is flagged if it normally runs at 35%. 

mean per server:
 [[65.8]
 [37.6]
 [94.2]]
std per server:
 [[17.17439955]
 [ 5.0039984 ]
 [ 3.86781592]]
z sore 
 [[-1.39 -0.63  0.71  1.47 -0.16]
 [-1.52 -0.52  0.48  0.08  1.48]
 [-1.6  -0.57  0.21  0.72  1.24]]
Anamoly detection on z-score:
 [[False False False False False]
 [False False False False False]
 [False False False False False]]


In [46]:
# Dataset
import numpy as np

servers = np.array([
    [55, 60, 78, 91, 85],   # server A
    [30, 32, 35, 33, 31],   # server B
    [70, 75, 88, 95, 99],   # server C
])

# Baselines per server — server A=60, B=32, C=75
baselines = np.array([[60], [32], [75]])
print("baselines \n", baselines)

latency = np.array([12, 15, 14, 45, 16, 13, 60, 18])
print("latency \n", latency)

# 1. Subtract each server's baseline from its own readings using broadcasting

diff = servers - baselines
print("Substract each servers baseline from its own reading\n", diff)

# 2. Normalise each server's readings to 0-1 range using per-server min and max
row_min = servers.min(axis=1, keepdims=True)
row_max = servers.max(axis=1, keepdims=True)
normalised = (servers - row_min) / (row_max - row_min)
print(normalised.round(2))

# 3. Calculate z-score per server flag readings more than 1.5 std above mean
mean = servers.mean(axis=1, keepdims=True)
std  = servers.std(axis=1, keepdims=True)
z    = (servers - mean)/std
print(z.round(2))
print(z > 1.5)

# 4. Subtract the overall mean of latency from every reading then print how far each reading sits from the mean
latency_mean = latency.mean()
print(latency_mean)
diff_latency = latency - latency_mean
print(diff_latency.round(1))

baselines 
 [[60]
 [32]
 [75]]
latency 
 [12 15 14 45 16 13 60 18]
Substract each servers baseline from its own reading
 [[-5  0 18 31 25]
 [-2  0  3  1 -1]
 [-5  0 13 20 24]]
[[0.   0.14 0.64 1.   0.83]
 [0.   0.4  1.   0.6  0.2 ]
 [0.   0.17 0.62 0.86 1.  ]]
[[-1.34 -0.98  0.3   1.23  0.8 ]
 [-1.28 -0.12  1.63  0.46 -0.7 ]
 [-1.37 -0.93  0.23  0.86  1.21]]
[[False False False False False]
 [False False  True False False]
 [False False False False False]]
24.125
[-12.1  -9.1 -10.1  20.9  -8.1 -11.1  35.9  -6.1]
